# 🤖 NewsBot Intelligence System
## ITAI 2373 - Mid-Term Group Project Template

**Project Type:** Individual project — [Your full name]
**Date:** June 24, 2026
**GitHub Repository:** https://github.com/viksus164/Midterm-ITAI2373

---

## 🎯 Project Overview

Welcome to your NewsBot Intelligence System! This notebook will guide you through building a comprehensive NLP system that:

- 📰 **Processes** news articles with advanced text cleaning
- 🏷️ **Classifies** articles into categories (Politics, Sports, Technology, Business, Entertainment, Health)
- 🔍 **Extracts** named entities (people, organizations, locations, dates, money)
- 😊 **Analyzes** sentiment and emotional tone
- 📊 **Generates** insights for business intelligence

### 📚 Module Integration Checklist
- [ ] **Module 1:** NLP applications and real-world context
- [ ] **Module 2:** Text preprocessing pipeline
- [ ] **Module 3:** TF-IDF feature extraction
- [ ] **Module 4:** POS tagging analysis
- [ ] **Module 5:** Syntax parsing and semantic analysis
- [ ] **Module 6:** Sentiment and emotion analysis
- [ ] **Module 7:** Text classification system
- [ ] **Module 8:** Named Entity Recognition

---


## 📦 Setup and Installation

Let's start by installing and importing all the libraries we'll need for our NewsBot system.

In [ ]:
# Install required packages (run this cell first!)
!pip install spacy scikit-learn nltk pandas matplotlib seaborn wordcloud plotly
!python -m spacy download en_core_web_sm

# Download NLTK data
import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('vader_lexicon')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('punkt_tab')

print("✅ All packages installed successfully!")


In [ ]:
# Import all necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import plotly.express as px
import plotly.graph_objects as go
from collections import Counter, defaultdict
import re
import warnings
warnings.filterwarnings('ignore')

# NLP Libraries
import spacy
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.sentiment import SentimentIntensityAnalyzer
from nltk.tag import pos_tag

# Scikit-learn for machine learning
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.svm import SVC, LinearSVC
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.pipeline import Pipeline
from scipy.sparse import hstack, csr_matrix

# Load spaCy model
nlp = spacy.load('en_core_web_sm')

# Set up plotting style
plt.style.use('default')
sns.set_palette("husl")

print("📚 All libraries imported successfully!")
print(f"🔧 spaCy model loaded: {nlp.meta['name']} v{nlp.meta['version']}")

## Data Loading and Exploration

This section loads the BBC News Classification dataset, standardizes the required columns, and verifies the project dataset rules: at least 500 English articles, at least 4 labeled categories, substantial article text, no missing text/category values, and fewer than 2,000 records for Colab performance.

The exploration also checks class balance, source information, text length, missing values, and duplicate/quality issues before any modeling is done.


In [ ]:
# Load the real BBC News Classification data. This replaces the template's ten-row sample.
# Default: upload the learn-ai-bbc.zip file you already downloaded.
# Optional: set USE_DOWNLOADED_ZIP to False to use the Kaggle API method from the assignment.
from pathlib import Path
import zipfile

USE_DOWNLOADED_ZIP = True
DATA_FOLDER = Path("bbc_data")
DATA_FOLDER.mkdir(exist_ok=True)

if USE_DOWNLOADED_ZIP:
    from google.colab import files
    print("Upload learn-ai-bbc.zip from your Downloads folder.")
    uploaded = files.upload()
    zip_name = next((name for name in uploaded if name.lower().endswith(".zip")), None)
    if zip_name is None:
        raise ValueError("Please upload learn-ai-bbc.zip.")
    with zipfile.ZipFile(zip_name) as archive:
        archive.extractall(DATA_FOLDER)
else:
    # Do not commit kaggle.json to GitHub.
    from google.colab import files
    print("Upload kaggle.json from your Kaggle account.")
    uploaded = files.upload()
    if "kaggle.json" not in uploaded:
        raise ValueError("Please upload kaggle.json.")
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    !kaggle competitions download -c learn-ai-bbc -p bbc_data
    !unzip -o bbc_data/learn-ai-bbc.zip -d bbc_data

raw_df = pd.read_csv(DATA_FOLDER / "BBC News Train.csv")
df = raw_df.rename(columns={"ArticleId": "article_id", "Text": "content", "Category": "category"})[
    ["article_id", "content", "category"]
].copy()
df["content"] = df["content"].fillna("").astype(str).str.strip()
df["category"] = df["category"].fillna("").astype(str).str.strip()
df["title"] = df["content"].str.split(r"\s{2,}|\n", n=1).str[0].str.split().str[:16].str.join(" ").str.title()
df["source"] = "BBC News Classification (Kaggle)"
df["date"] = pd.NaT  # The supplied file has no verified publication dates.
df = df[(df["content"].str.len() >= 200) & (df["category"].str.len() > 0)].copy()

requirements = {
    "At least 500 articles": len(df) >= 500,
    "At least 4 categories": df["category"].nunique() >= 4,
    "At most 2,000 articles": len(df) <= 2000,
    "No missing text": df["content"].notna().all() and df["content"].str.len().gt(0).all(),
    "No missing category": df["category"].notna().all() and df["category"].str.len().gt(0).all(),
    "Substantial article text": df["content"].str.len().ge(200).all(),
}
display(pd.DataFrame({"Requirement": requirements.keys(), "Passed": requirements.values()}))
assert all(requirements.values()), "Dataset validation failed."
print(f"Loaded {len(df)} English BBC articles in {df['category'].nunique()} categories.")
display(df.head())


In [ ]:
# Basic dataset exploration
print("DATASET OVERVIEW")
print("=" * 50)
print(f"Total articles: {len(df)}")
print(f"Unique categories: {df['category'].nunique()}")
print(f"Categories: {df['category'].unique().tolist()}")

if df['date'].notna().any():
    print(f"Date range: {df['date'].min()} to {df['date'].max()}")
else:
    print("Date range: Not available in the BBC Kaggle training file")

print(f"Unique sources: {df['source'].nunique()}")

print("\nCATEGORY DISTRIBUTION")
print("=" * 50)
category_counts = df['category'].value_counts()
print(category_counts)

# Completed exploratory analysis: missing values, text length, source distribution, and quality checks.
print("\nDATA QUALITY CHECKS")
print("=" * 50)
missing_summary = df[['article_id', 'title', 'content', 'category', 'source', 'date']].isna().sum()
print("Missing values by column:")
print(missing_summary)
print("Note: date is missing because the BBC Kaggle training file does not include verified publication dates; this is handled later through temporal-reference analysis.")

df['content_char_length'] = df['content'].astype(str).str.len()
df['content_word_count'] = df['content'].astype(str).str.split().str.len()
df['title_word_count'] = df['title'].astype(str).str.split().str.len()

print("\nArticle length summary:")
print(df[['content_char_length', 'content_word_count', 'title_word_count']].describe().round(2))

source_counts = df['source'].value_counts()
print("\nSource distribution:")
print(source_counts)

duplicate_articles = df.duplicated(subset=['content']).sum()
short_articles = (df['content_word_count'] < 50).sum()
empty_titles = (df['title'].astype(str).str.strip() == '').sum()

quality_summary = pd.DataFrame({
    'Check': [
        'Duplicate article bodies',
        'Articles under 50 words',
        'Empty derived titles',
        'Missing category values',
        'Missing content values'
    ],
    'Count': [
        duplicate_articles,
        short_articles,
        empty_titles,
        df['category'].isna().sum(),
        df['content'].isna().sum()
    ]
})
print("\nQuality issue summary:")
print(quality_summary)

requirement_check = pd.DataFrame({
    'Requirement': [
        'At least 500 articles',
        'At least 4 categories',
        'At most 2,000 articles',
        'No missing article text',
        'No missing category labels',
        'Substantial article text'
    ],
    'Passed': [
        len(df) >= 500,
        df['category'].nunique() >= 4,
        len(df) <= 2000,
        df['content'].notna().all(),
        df['category'].notna().all(),
        (df['content_word_count'] >= 50).all()
    ]
})
print("\nDataset requirement confirmation:")
print(requirement_check)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.countplot(data=df, x='category', order=category_counts.index, ax=axes[0])
axes[0].set_title('Distribution of News Categories')
axes[0].tick_params(axis='x', rotation=45)

sns.histplot(df['content_word_count'], bins=30, kde=True, ax=axes[1])
axes[1].set_title('Article Word Count Distribution')
axes[1].set_xlabel('Words per article')

source_counts.plot(kind='bar', ax=axes[2])
axes[2].set_title('Source Distribution')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


## Text Preprocessing Pipeline

This section turns raw article text into model-ready text. The cleaning pipeline removes HTML, URLs, email addresses, punctuation/numbers, extra spaces, stop words, and applies lemmatization.

The output keeps both the original article text and processed text so later sections can compare human-readable examples with the cleaned features used by TF-IDF and machine-learning models.


In [ ]:
# Initialize preprocessing tools
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
    """
    Clean raw article text for NLP processing.

    The function removes HTML tags, URLs, email addresses, digits,
    punctuation/special characters, and extra whitespace. It returns
    lowercase alphabetic text that is easier to tokenize and model.
    """
    if pd.isna(text):
        return ""
    
    text = str(text).lower()
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

def preprocess_text(text, remove_stopwords=True, lemmatize=True):
    """
    Run the full preprocessing pipeline on a text string.

    Steps:
    1. Clean the raw text.
    2. Tokenize into words.
    3. Optionally remove English stop words.
    4. Optionally lemmatize words.
    5. Remove very short tokens.

    Returns a single processed string suitable for TF-IDF modeling.
    """
    text = clean_text(text)
    
    if not text:
        return ""
    
    tokens = word_tokenize(text)
    
    if remove_stopwords:
        tokens = [token for token in tokens if token not in stop_words]
    
    if lemmatize:
        tokens = [lemmatizer.lemmatize(token) for token in tokens]
    
    tokens = [token for token in tokens if len(token) > 2]
    
    return ' '.join(tokens)

sample_text = "Apple Inc. announced record quarterly earnings today! Visit https://apple.com for more info. #TechNews"
print("Original text:")
print(sample_text)
print("\nCleaned text:")
print(clean_text(sample_text))
print("\nFully preprocessed text:")
print(preprocess_text(sample_text))


In [ ]:
# Apply preprocessing to the dataset
print("Preprocessing all articles...")

df['title_clean'] = df['title'].apply(clean_text)
df['content_clean'] = df['content'].apply(clean_text)
df['title_processed'] = df['title'].apply(preprocess_text)
df['content_processed'] = df['content'].apply(preprocess_text)

df['full_text'] = df['title'] + ' ' + df['content']
df['full_text_processed'] = df['full_text'].apply(preprocess_text)

print("Preprocessing complete!")

print("\nBEFORE AND AFTER EXAMPLES")
print("=" * 60)
for i in range(min(3, len(df))):
    print(f"\nExample {i+1}:")
    print(f"Original: {df.iloc[i]['full_text'][:100]}...")
    print(f"Processed: {df.iloc[i]['full_text_processed'][:100]}...")

# Completed preprocessing analysis: length reduction, vocabulary change, and most common processed words.
df['raw_word_count'] = df['full_text'].astype(str).str.split().str.len()
df['processed_word_count'] = df['full_text_processed'].astype(str).str.split().str.len()
df['preprocessing_reduction_pct'] = (1 - (df['processed_word_count'] / df['raw_word_count'].replace(0, np.nan))) * 100

raw_tokens = ' '.join(df['full_text'].apply(clean_text)).split()
processed_tokens = ' '.join(df['full_text_processed']).split()

preprocessing_summary = pd.DataFrame({
    'Metric': [
        'Average raw words per article',
        'Average processed words per article',
        'Average word reduction (%)',
        'Unique raw words',
        'Unique processed words'
    ],
    'Value': [
        df['raw_word_count'].mean(),
        df['processed_word_count'].mean(),
        df['preprocessing_reduction_pct'].mean(),
        len(set(raw_tokens)),
        len(set(processed_tokens))
    ]
})
print("\nPREPROCESSING SUMMARY")
print(preprocessing_summary.round(2))

common_processed_words = pd.DataFrame(Counter(processed_tokens).most_common(20), columns=['word', 'count'])
print("\nMost common processed words:")
print(common_processed_words)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

length_compare = df[['raw_word_count', 'processed_word_count']].melt(var_name='Text version', value_name='Word count')
sns.boxplot(data=length_compare, x='Text version', y='Word count', ax=axes[0])
axes[0].set_title('Raw vs Processed Article Length')

sns.barplot(data=common_processed_words.head(15), x='count', y='word', ax=axes[1])
axes[1].set_title('Top 15 Words After Preprocessing')
axes[1].set_xlabel('Frequency')
axes[1].set_ylabel('Word')

plt.tight_layout()
plt.show()


## Feature Extraction and Statistical Analysis

This section converts articles into TF-IDF features. TF-IDF highlights words and phrases that are important within a category while reducing the weight of very common words.

The analysis identifies the most important terms for each news category and visualizes category-specific language patterns.


In [ ]:
# Create TF-IDF vectorizer
# 💡 TIP: Experiment with different parameters:
# - max_features: limit vocabulary size
# - ngram_range: include phrases (1,1) for words, (1,2) for words+bigrams
# - min_df: ignore terms that appear in less than min_df documents
# - max_df: ignore terms that appear in more than max_df fraction of documents

tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,  # Limit vocabulary for computational efficiency
    ngram_range=(1, 2),  # Include unigrams and bigrams
    min_df=2,  # Ignore terms that appear in less than 2 documents
    max_df=0.8  # Ignore terms that appear in more than 80% of documents
)

# Fit and transform the processed text
print("🔢 Creating TF-IDF features...")
tfidf_matrix = tfidf_vectorizer.fit_transform(df['full_text_processed'])
feature_names = tfidf_vectorizer.get_feature_names_out()

print(f"✅ TF-IDF matrix created!")
print(f"📊 Shape: {tfidf_matrix.shape}")
print(f"📝 Vocabulary size: {len(feature_names)}")
print(f"🔢 Sparsity: {(1 - tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1])) * 100:.2f}%")

# Convert to DataFrame for easier analysis
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=feature_names)
tfidf_df['category'] = df['category'].values

print("\n🔍 Sample TF-IDF features:")
print(tfidf_df.iloc[:3, :10])  # Show first 3 rows and 10 features

In [ ]:
# Analyze most important terms per category
def get_top_tfidf_terms(category, n_terms=10):
    """
    Return the top TF-IDF terms for one news category.

    The function filters rows for the selected category, calculates the
    mean TF-IDF score for every vocabulary term, and returns the highest
    scoring terms. Higher scores indicate words/phrases that are more
    distinctive for that category.
    """
    category_data = tfidf_df[tfidf_df['category'] == category]
    mean_scores = category_data.drop('category', axis=1).mean().sort_values(ascending=False)
    return mean_scores.head(n_terms)

print("TOP TF-IDF TERMS BY CATEGORY")
print("=" * 50)

categories = df['category'].unique()
category_terms = {}

for category in categories:
    top_terms = get_top_tfidf_terms(category, n_terms=10)
    category_terms[category] = top_terms
    
    print(f"\n{category.upper()}:")
    for term, score in top_terms.items():
        print(f"  {term}: {score:.4f}")

# Completed TF-IDF visualizations: category bar charts, heatmap, and word clouds.
top_term_records = []
for category, terms in category_terms.items():
    for term, score in terms.items():
        top_term_records.append({'category': category, 'term': term, 'tfidf_score': score})

top_terms_df = pd.DataFrame(top_term_records)

n_cols = 2
n_rows = int(np.ceil(len(categories) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
axes = np.array(axes).reshape(-1)

for ax, category in zip(axes, categories):
    plot_data = top_terms_df[top_terms_df['category'] == category].sort_values('tfidf_score', ascending=True)
    ax.barh(plot_data['term'], plot_data['tfidf_score'])
    ax.set_title(f'Top TF-IDF Terms: {category}')
    ax.set_xlabel('Mean TF-IDF score')

for ax in axes[len(categories):]:
    ax.axis('off')

plt.tight_layout()
plt.show()

important_terms = list(dict.fromkeys(top_terms_df['term'].tolist()))[:30]
heatmap_rows = []
for category in categories:
    category_data = tfidf_df[tfidf_df['category'] == category]
    heatmap_rows.append(category_data[important_terms].mean())

tfidf_heatmap = pd.DataFrame(heatmap_rows, index=categories)
plt.figure(figsize=(16, 6))
sns.heatmap(tfidf_heatmap, cmap='YlGnBu')
plt.title('TF-IDF Term Importance Across Categories')
plt.xlabel('Important terms')
plt.ylabel('Category')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
axes = np.array(axes).reshape(-1)

for ax, category in zip(axes, categories):
    frequencies = category_terms[category].to_dict()
    wordcloud = WordCloud(width=800, height=400, background_color='white', colormap='viridis').generate_from_frequencies(frequencies)
    ax.imshow(wordcloud, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(f'Word Cloud: {category}')

for ax in axes[len(categories):]:
    ax.axis('off')

plt.tight_layout()
plt.show()


## Part-of-Speech Analysis

This section uses POS tagging to measure how each category uses nouns, verbs, adjectives, adverbs, and numbers. These patterns help show linguistic differences between news categories beyond simple keyword counts.


In [ ]:
def analyze_pos_patterns(text):
    """
    Analyze part-of-speech patterns in one article.

    The function tokenizes the article, applies NLTK POS tagging, counts
    each POS tag, and returns proportions so articles of different lengths
    can be compared fairly.
    """
    if not text or pd.isna(text):
        return {}
    
    tokens = word_tokenize(str(text))
    pos_tags = pos_tag(tokens)
    pos_counts = Counter([tag for word, tag in pos_tags])
    total_words = len(pos_tags)
    
    if total_words == 0:
        return {}
    
    pos_proportions = {pos: count / total_words for pos, count in pos_counts.items()}
    return pos_proportions

print("Analyzing POS patterns...")

pos_results = []
for idx, row in df.iterrows():
    pos_analysis = analyze_pos_patterns(row['full_text'])
    pos_analysis['category'] = row['category']
    pos_analysis['article_id'] = row['article_id']
    pos_results.append(pos_analysis)

pos_df = pd.DataFrame(pos_results).fillna(0)

print("POS analysis complete!")
print(f"Found {len(pos_df.columns)-2} different POS tags")

print("\nSample POS analysis:")
print(pos_df.head())


In [ ]:
# Analyze POS patterns by category
print("POS PATTERNS BY CATEGORY")
print("=" * 50)

pos_by_category = pos_df.groupby('category').mean(numeric_only=True)

major_pos = ['NN', 'NNS', 'NNP', 'NNPS', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ',
             'JJ', 'JJR', 'JJS', 'RB', 'RBR', 'RBS', 'CD']

available_pos = [pos for pos in major_pos if pos in pos_by_category.columns]

if available_pos:
    pos_summary = pos_by_category[available_pos]
    
    print("\nKey POS patterns by category:")
    print(pos_summary.round(4))
    
    plt.figure(figsize=(12, 8))
    sns.heatmap(pos_summary.T, annot=True, cmap='YlOrRd', fmt='.3f')
    plt.title('POS Tag Proportions by News Category')
    plt.xlabel('Category')
    plt.ylabel('POS Tag')
    plt.tight_layout()
    plt.show()
    
    # Completed POS interpretation: nouns vs verbs, numbers, and adjectives.
    noun_tags = [tag for tag in ['NN', 'NNS', 'NNP', 'NNPS'] if tag in pos_by_category.columns]
    verb_tags = [tag for tag in ['VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ'] if tag in pos_by_category.columns]
    proper_noun_tags = [tag for tag in ['NNP', 'NNPS'] if tag in pos_by_category.columns]
    adjective_tags = [tag for tag in ['JJ', 'JJR', 'JJS'] if tag in pos_by_category.columns]
    number_tags = [tag for tag in ['CD'] if tag in pos_by_category.columns]
    
    pos_interpretation = pd.DataFrame(index=pos_by_category.index)
    pos_interpretation['noun_proportion'] = pos_by_category[noun_tags].sum(axis=1) if noun_tags else 0
    pos_interpretation['verb_proportion'] = pos_by_category[verb_tags].sum(axis=1) if verb_tags else 0
    pos_interpretation['proper_noun_proportion'] = pos_by_category[proper_noun_tags].sum(axis=1) if proper_noun_tags else 0
    pos_interpretation['adjective_proportion'] = pos_by_category[adjective_tags].sum(axis=1) if adjective_tags else 0
    pos_interpretation['number_proportion'] = pos_by_category[number_tags].sum(axis=1) if number_tags else 0
    pos_interpretation['noun_to_verb_ratio'] = pos_interpretation['noun_proportion'] / pos_interpretation['verb_proportion'].replace(0, np.nan)
    
    print("\nPOS interpretation summary:")
    print(pos_interpretation.round(4))
    
    highest_proper = pos_interpretation['proper_noun_proportion'].idxmax()
    highest_verb = pos_interpretation['verb_proportion'].idxmax()
    highest_adj = pos_interpretation['adjective_proportion'].idxmax()
    highest_number = pos_interpretation['number_proportion'].idxmax()
    
    print("\nWritten POS findings:")
    print(f"1. Highest proper noun usage: {highest_proper}")
    print(f"2. Highest action verb usage: {highest_verb}")
    print(f"3. Highest adjective usage: {highest_adj}")
    print(f"4. Highest number/cardinal usage: {highest_number}")
    if 'business' in pos_interpretation.index:
        business_rank = pos_interpretation['number_proportion'].rank(ascending=False, method='min').loc['business']
        print(f"5. Business number usage rank: #{int(business_rank)} out of {len(pos_interpretation)} categories")
    
    pos_interpretation[['noun_proportion', 'verb_proportion', 'adjective_proportion', 'number_proportion']].plot(
        kind='bar', figsize=(12, 6)
    )
    plt.title('Grouped POS Patterns by Category')
    plt.ylabel('Average proportion')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("No major POS tags found in the analysis. Check your POS tagging implementation.")


## Syntax Parsing and Semantic Analysis

This section uses spaCy dependency parsing to examine sentence structure. It extracts syntactic complexity, noun phrases, subject/object patterns, and simple action patterns such as who did what.


In [ ]:
def extract_syntactic_features(text):
    """
    Extract syntactic features with spaCy dependency parsing.

    The function returns sentence counts, token counts, dependency labels,
    noun phrases, verb lemmas, subjects, objects, and simple action patterns
    that approximate who did what in an article.
    """
    if not text or pd.isna(text):
        return {}
    
    doc = nlp(str(text))
    sentence_lengths = [len([token for token in sent if not token.is_punct and not token.is_space]) for sent in doc.sents]
    
    features = {
        'num_sentences': len(sentence_lengths),
        'num_tokens': len([token for token in doc if not token.is_space and not token.is_punct]),
        'avg_sentence_length': np.mean(sentence_lengths) if sentence_lengths else 0,
        'dependency_relations': [],
        'noun_phrases': [],
        'verb_phrases': [],
        'subjects': [],
        'objects': [],
        'action_patterns': []
    }
    
    for token in doc:
        if not token.is_space and not token.is_punct:
            features['dependency_relations'].append(token.dep_)
            
            if token.pos_ == 'VERB':
                features['verb_phrases'].append(token.lemma_.lower())
            
            if token.dep_ in ['nsubj', 'nsubjpass']:
                features['subjects'].append(token.text.lower())
                verb = token.head
                objects = [child.text.lower() for child in verb.children if child.dep_ in ['dobj', 'iobj', 'pobj', 'attr', 'dative']]
                features['action_patterns'].append({
                    'subject': token.text.lower(),
                    'verb': verb.lemma_.lower(),
                    'objects': objects
                })
            elif token.dep_ in ['dobj', 'iobj', 'pobj', 'attr', 'dative']:
                features['objects'].append(token.text.lower())
    
    for chunk in doc.noun_chunks:
        features['noun_phrases'].append(chunk.text.lower())
    
    dep_counts = Counter(features['dependency_relations'])
    features['dependency_counts'] = dict(dep_counts)
    return features

print("Performing syntactic analysis...")

syntactic_results = []
for idx, row in df.head(5).iterrows():
    features = extract_syntactic_features(row['full_text'][:3000])
    features['category'] = row['category']
    features['article_id'] = row['article_id']
    syntactic_results.append(features)

print("Syntactic analysis complete!")

for i, result in enumerate(syntactic_results):
    print(f"\nArticle {i+1} ({result['category']}):")
    print(f"  Sentences: {result['num_sentences']}")
    print(f"  Tokens: {result['num_tokens']}")
    print(f"  Avg sentence length: {result['avg_sentence_length']:.1f}")
    print(f"  Noun phrases: {result['noun_phrases'][:3]}...")
    print(f"  Verb phrases: {result['verb_phrases'][:3]}...")
    print(f"  Subjects: {result['subjects'][:3]}...")
    print(f"  Objects: {result['objects'][:3]}...")


In [ ]:
# Visualize dependency parsing for a sample sentence
from spacy import displacy

sentences = sent_tokenize(df.iloc[0]['content'])
sample_sentence = sentences[0] if sentences else df.iloc[0]['content'][:500]
print(f"Sample sentence: {sample_sentence}")

doc = nlp(sample_sentence)

print("\nDependency Parse Visualization:")
try:
    displacy.render(doc, style="dep", jupyter=True)
except Exception:
    print("\nDependency Relations:")
    for token in doc:
        if not token.is_space and not token.is_punct:
            print(f"  {token.text} --> {token.dep_} --> {token.head.text}")

# Completed syntactic extension: compare complexity, dependency relations, and action patterns across categories.
print("\nSYNTACTIC COMPLEXITY BY CATEGORY")
print("=" * 50)

syntax_sample = df.groupby('category', group_keys=False).head(5)
syntax_rows = []
dependency_by_category = defaultdict(Counter)
action_pattern_rows = []

for _, row in syntax_sample.iterrows():
    features = extract_syntactic_features(row['full_text'][:3000])
    syntax_rows.append({
        'category': row['category'],
        'article_id': row['article_id'],
        'num_sentences': features.get('num_sentences', 0),
        'num_tokens': features.get('num_tokens', 0),
        'avg_sentence_length': features.get('avg_sentence_length', 0),
        'noun_phrase_count': len(features.get('noun_phrases', [])),
        'verb_count': len(features.get('verb_phrases', [])),
        'subject_count': len(features.get('subjects', [])),
        'object_count': len(features.get('objects', []))
    })
    dependency_by_category[row['category']].update(features.get('dependency_counts', {}))
    for pattern in features.get('action_patterns', [])[:5]:
        action_pattern_rows.append({
            'category': row['category'],
            'subject': pattern['subject'],
            'verb': pattern['verb'],
            'objects': ', '.join(pattern['objects']) if pattern['objects'] else '(none)'
        })

syntax_df = pd.DataFrame(syntax_rows)
syntax_summary = syntax_df.groupby('category').mean(numeric_only=True).round(2)
print("\nAverage syntactic features by category:")
print(syntax_summary)

print("\nMost common dependency relations by category:")
for category, counter in dependency_by_category.items():
    print(f"\n{category}:")
    for dep, count in counter.most_common(5):
        print(f"  {dep}: {count}")

if action_pattern_rows:
    action_patterns_df = pd.DataFrame(action_pattern_rows)
    print("\nSample action patterns (who did what):")
    print(action_patterns_df.head(10))
else:
    action_patterns_df = pd.DataFrame(columns=['category', 'subject', 'verb', 'objects'])
    print("\nNo action patterns extracted from the sample.")

syntax_summary[['avg_sentence_length', 'noun_phrase_count', 'verb_count', 'subject_count', 'object_count']].plot(
    kind='bar', figsize=(12, 6)
)
plt.title('Syntactic Complexity Features by Category')
plt.ylabel('Average value in sampled articles')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## Sentiment and Emotion Analysis

This section applies VADER sentiment analysis to headlines, article bodies, and combined article text. The goal is to compare emotional tone across categories and create sentiment features for classification.


In [ ]:
# Initialize sentiment analyzer
sia = SentimentIntensityAnalyzer()

def analyze_sentiment(text):
    """
    Analyze sentiment with VADER.

    Returns the compound sentiment score, positive/neutral/negative scores,
    and a readable label: positive, negative, or neutral.
    """
    if not text or pd.isna(text):
        return {'compound': 0, 'pos': 0, 'neu': 1, 'neg': 0, 'sentiment_label': 'neutral'}
    
    scores = sia.polarity_scores(str(text))
    
    if scores['compound'] >= 0.05:
        scores['sentiment_label'] = 'positive'
    elif scores['compound'] <= -0.05:
        scores['sentiment_label'] = 'negative'
    else:
        scores['sentiment_label'] = 'neutral'
    
    return scores

print("Analyzing sentiment...")

sentiment_results = []
for idx, row in df.iterrows():
    title_sentiment = analyze_sentiment(row['title'])
    content_sentiment = analyze_sentiment(row['content'])
    full_sentiment = analyze_sentiment(row['full_text'])
    
    result = {
        'article_id': row['article_id'],
        'category': row['category'],
        'title_sentiment': title_sentiment['compound'],
        'title_label': title_sentiment['sentiment_label'],
        'content_sentiment': content_sentiment['compound'],
        'content_label': content_sentiment['sentiment_label'],
        'full_sentiment': full_sentiment['compound'],
        'full_label': full_sentiment['sentiment_label'],
        'pos_score': full_sentiment['pos'],
        'neu_score': full_sentiment['neu'],
        'neg_score': full_sentiment['neg']
    }
    sentiment_results.append(result)

sentiment_df = pd.DataFrame(sentiment_results)

print("Sentiment analysis complete!")
print(f"Analyzed {len(sentiment_df)} articles")

print("\nSample sentiment results:")
print(sentiment_df[['category', 'full_sentiment', 'full_label']].head())


In [ ]:
# Analyze sentiment patterns by category
print("SENTIMENT ANALYSIS BY CATEGORY")
print("=" * 50)

sentiment_by_category = sentiment_df.groupby('category').agg({
    'full_sentiment': ['mean', 'std', 'min', 'max'],
    'pos_score': 'mean',
    'neu_score': 'mean',
    'neg_score': 'mean'
}).round(4)

print("\nSentiment statistics by category:")
print(sentiment_by_category)

sentiment_dist = sentiment_df.groupby(['category', 'full_label']).size().unstack(fill_value=0)
sentiment_dist_pct = sentiment_dist.div(sentiment_dist.sum(axis=1), axis=0) * 100

print("\nSentiment distribution (%) by category:")
print(sentiment_dist_pct.round(2))

fig, axes = plt.subplots(2, 2, figsize=(15, 12))

sns.boxplot(data=sentiment_df, x='category', y='full_sentiment', ax=axes[0,0])
axes[0,0].set_title('Sentiment Score Distribution by Category')
axes[0,0].tick_params(axis='x', rotation=45)

sentiment_dist_pct.plot(kind='bar', ax=axes[0,1], stacked=True)
axes[0,1].set_title('Sentiment Label Distribution by Category (%)')
axes[0,1].tick_params(axis='x', rotation=45)
axes[0,1].legend(title='Sentiment')

category_means = sentiment_df.groupby('category')[['pos_score', 'neg_score']].mean()
category_means.plot(kind='bar', ax=axes[1,0])
axes[1,0].set_title('Average Positive vs Negative Scores by Category')
axes[1,0].tick_params(axis='x', rotation=45)
axes[1,0].legend(['Positive', 'Negative'])

sentiment_pivot = sentiment_df.pivot_table(values='full_sentiment', index='category',
                                           columns='full_label', aggfunc='count', fill_value=0)
sns.heatmap(sentiment_pivot, annot=True, fmt='d', ax=axes[1,1], cmap='YlOrRd')
axes[1,1].set_title('Sentiment Count Heatmap')

plt.tight_layout()
plt.show()

# Completed sentiment interpretation.
mean_sentiment = sentiment_df.groupby('category')['full_sentiment'].mean().sort_values(ascending=False)
sentiment_variability = sentiment_df.groupby('category')['full_sentiment'].std().sort_values(ascending=False)
title_content_gap = sentiment_df.assign(
    title_content_difference=sentiment_df['content_sentiment'] - sentiment_df['title_sentiment']
).groupby('category')['title_content_difference'].mean().sort_values(ascending=False)
label_mismatch_rate = (sentiment_df['title_label'] != sentiment_df['content_label']).mean()

print("\nWritten sentiment findings:")
print(f"1. Most positive category by average compound score: {mean_sentiment.index[0]} ({mean_sentiment.iloc[0]:.4f})")
print(f"2. Most negative category by average compound score: {mean_sentiment.index[-1]} ({mean_sentiment.iloc[-1]:.4f})")
print(f"3. Category with the most sentiment variation: {sentiment_variability.index[0]} ({sentiment_variability.iloc[0]:.4f} std)")
print(f"4. Title/content sentiment labels differ in {label_mismatch_rate:.1%} of articles.")
print("5. Sentiment can be used as a classification feature; Module 7 uses scaled compound, positive, neutral, and negative scores.")

print("\nAverage content-minus-title sentiment difference by category:")
print(title_content_gap.round(4))


## Text Classification System

This section builds the main NewsBot classifier. The model combines TF-IDF text features with scaled sentiment and length features, compares several algorithms, tunes Logistic Regression, and selects the best-performing model.


In [ ]:
# Prepare features for classification
print("Preparing features for classification...")

# IMPROVEMENT: keep TF-IDF as a sparse matrix instead of converting it to a huge dense array.
# This is faster, uses less memory, and works better in free Google Colab.
X_tfidf = tfidf_matrix

# Add sentiment features.
# VADER compound sentiment ranges from -1 to 1, so scale it to 0 to 1 for Naive Bayes.
compound_scaled = ((sentiment_df['full_sentiment'] + 1) / 2).values.reshape(-1, 1)
sentiment_features = np.hstack([
    compound_scaled,
    sentiment_df[['pos_score', 'neu_score', 'neg_score']].values
])

# Add text length features and scale them so they do not overpower TF-IDF.
length_features = np.array([
    df['full_text'].str.len(),
    df['full_text'].str.split().str.len(),
    df['title'].str.len(),
]).T.astype(float)
length_feature_max = np.maximum(length_features.max(axis=0), 1)
length_features = length_features / length_feature_max

# Combine TF-IDF + sentiment + length features.
# This satisfies the feature engineering requirement while preserving computational efficiency.
extra_features = csr_matrix(np.hstack([sentiment_features, length_features]))
X_combined = hstack([X_tfidf, extra_features]).tocsr()

# Target variable
y = df['category'].values

print("Feature matrix prepared!")
print(f"Feature matrix shape: {X_combined.shape}")
print(f"Number of classes: {len(np.unique(y))}")
print(f"Classes: {np.unique(y)}")

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X_combined, y, test_size=0.2, random_state=42, stratify=y
)

print("\nData split:")
print(f"  Training set: {X_train.shape[0]} samples")
print(f"  Test set: {X_test.shape[0]} samples")


In [ ]:
# Train and evaluate multiple classifiers
print("Training multiple improved classifiers...")

# IMPROVEMENT 1: Hyperparameter tuning for Logistic Regression.
# We test a few C values and keep the one with the best cross-validation score.
print("\nTuning Logistic Regression C value...")
c_values = [0.5, 1.0, 2.0, 4.0]
tuning_results = []

for c in c_values:
    candidate = LogisticRegression(
        random_state=42,
        max_iter=1000,
        solver='liblinear',
        C=c,
        class_weight='balanced'
    )
    scores = cross_val_score(candidate, X_train, y_train, cv=3, scoring='accuracy')
    tuning_results.append({'C': c, 'CV Mean': scores.mean(), 'CV Std': scores.std()})
    print(f"  C={c}: CV accuracy={scores.mean():.4f}")

tuning_df = pd.DataFrame(tuning_results)
best_c = tuning_df.loc[tuning_df['CV Mean'].idxmax(), 'C']
print(f"Best C value selected: {best_c}")

# IMPROVEMENT 2: Compare stronger, Colab-friendly classifiers.
# - MultinomialNB is the baseline.
# - ComplementNB often works better for text classification.
# - Logistic Regression uses tuned C and balanced class weights.
# - LinearSVC is much faster and more appropriate for TF-IDF text than default SVC.
classifiers = {
    'Baseline Multinomial NB': MultinomialNB(alpha=0.5),
    'Improved Complement NB': ComplementNB(alpha=0.5),
    'Tuned Logistic Regression': LogisticRegression(
        random_state=42,
        max_iter=1000,
        solver='liblinear',
        C=best_c,
        class_weight='balanced'
    ),
    'Fast Linear SVM': LinearSVC(random_state=42, C=1.0, class_weight='balanced')
}

results = {}
trained_models = {}

for name, classifier in classifiers.items():
    print(f"\nTraining {name}...")
    
    classifier.fit(X_train, y_train)
    y_pred = classifier.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    
    # Cross-validation gives a more reliable estimate than one train/test split.
    cv_scores = cross_val_score(classifier, X_train, y_train, cv=3, scoring='accuracy')
    
    # Store probabilities when the model supports them.
    y_pred_proba = classifier.predict_proba(X_test) if hasattr(classifier, 'predict_proba') else None
    
    results[name] = {
        'accuracy': accuracy,
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std(),
        'predictions': y_pred,
        'probabilities': y_pred_proba
    }
    trained_models[name] = classifier
    
    print(f"  Test accuracy: {accuracy:.4f}")
    print(f"  CV Score: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")

print("\nCLASSIFIER COMPARISON")
print("=" * 50)
comparison_df = pd.DataFrame({
    'Model': list(results.keys()),
    'Test Accuracy': [results[name]['accuracy'] for name in results.keys()],
    'CV Mean': [results[name]['cv_mean'] for name in results.keys()],
    'CV Std': [results[name]['cv_std'] for name in results.keys()]
}).sort_values(by='Test Accuracy', ascending=False)

print(comparison_df.round(4))

# Find best model
best_model_name = comparison_df.iloc[0]['Model']
print(f"\nBest performing model: {best_model_name}")


In [ ]:
# Detailed evaluation of the best model
best_model = trained_models[best_model_name]
best_predictions = results[best_model_name]['predictions']

print(f"DETAILED EVALUATION: {best_model_name}")
print("=" * 60)

# Classification report
print("\nClassification Report:")
print(classification_report(y_test, best_predictions))

# Confusion matrix
cm = confusion_matrix(y_test, best_predictions)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=np.unique(y), yticklabels=np.unique(y))
plt.title(f'Confusion Matrix - {best_model_name}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

# Feature importance for linear models that expose coefficients
if hasattr(best_model, 'coef_'):
    print("\nTop Features by Category:")
    feature_names_extended = list(feature_names) + [
        'compound_sentiment_scaled', 'pos_score', 'neu_score', 'neg_score',
        'char_length_scaled', 'word_count_scaled', 'title_length_scaled'
    ]
    
    classes = best_model.classes_
    coefficients = best_model.coef_
    
    for i, class_name in enumerate(classes):
        top_indices = np.argsort(coefficients[i])[-10:]
        print(f"\n{class_name}:")
        for idx in reversed(top_indices):
            if idx < len(feature_names_extended):
                print(f"  {feature_names_extended[idx]}: {coefficients[i][idx]:.4f}")
else:
    print("\nFeature importance is skipped because this model does not expose coefficients.")

print("\nCLASSIFIER IMPROVEMENT SUMMARY")
print("- Used sparse TF-IDF features for better speed and memory efficiency in Colab.")
print("- Added scaled sentiment and length features to enrich the text representation.")
print("- Tuned Logistic Regression with cross-validation over multiple C values.")
print("- Added Complement Naive Bayes, which is often stronger for text classification.")
print("- Replaced slow default SVC with fast Linear SVM for TF-IDF text data.")
print("- Used stratified train/test split and class_weight='balanced' where supported.")


## Named Entity Recognition

This section extracts people, organizations, locations, dates, money amounts, and other named entities using spaCy. Entity analysis helps turn raw articles into media-monitoring intelligence about who and what appears in the news.


In [ ]:
def extract_entities(text):
    """
    Extract named entities using spaCy.

    Returns a list of dictionaries containing the entity text, label,
    character span, and a plain-English label description.
    """
    if not text or pd.isna(text):
        return []
    
    doc = nlp(str(text))
    
    entities = []
    for ent in doc.ents:
        entities.append({
            'text': ent.text,
            'label': ent.label_,
            'start': ent.start_char,
            'end': ent.end_char,
            'description': spacy.explain(ent.label_)
        })
    
    return entities

print("Extracting named entities...")

all_entities = []
article_entities = []

for idx, row in df.iterrows():
    entities = extract_entities(row['full_text'])
    
    article_entities.append({
        'article_id': row['article_id'],
        'category': row['category'],
        'entities': entities,
        'entity_count': len(entities)
    })
    
    for entity in entities:
        entity['article_id'] = row['article_id']
        entity['category'] = row['category']
        all_entities.append(entity)

print("Entity extraction complete!")
print(f"Total entities found: {len(all_entities)}")
print(f"Articles processed: {len(article_entities)}")

entities_df = pd.DataFrame(all_entities)

if not entities_df.empty:
    print(f"\nEntity types found: {entities_df['label'].unique()}")
    print("\nSample entities:")
    print(entities_df[['text', 'label', 'category']].head(10))
else:
    print("No entities found. This might happen with very short sample texts.")


In [ ]:
# Analyze entity patterns
from itertools import combinations

if not entities_df.empty:
    print("NAMED ENTITY ANALYSIS")
    print("=" * 50)
    
    entity_counts = entities_df['label'].value_counts()
    print("\nEntity type distribution:")
    print(entity_counts)
    
    entity_by_category = entities_df.groupby(['category', 'label']).size().unstack(fill_value=0)
    print("\nEntity types by news category:")
    print(entity_by_category)
    
    print("\nMost frequent entities:")
    frequent_entities = entities_df.groupby(['text', 'label']).size().sort_values(ascending=False).head(15)
    for (entity, label), count in frequent_entities.items():
        print(f"  {entity} ({label}): {count} mentions")
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    entity_counts.plot(kind='bar', ax=axes[0,0])
    axes[0,0].set_title('Entity Type Distribution')
    axes[0,0].tick_params(axis='x', rotation=45)
    
    entities_per_category = entities_df.groupby('category').size()
    entities_per_category.plot(kind='bar', ax=axes[0,1])
    axes[0,1].set_title('Total Entities per Category')
    axes[0,1].tick_params(axis='x', rotation=45)
    
    if entity_by_category.shape[0] > 1 and entity_by_category.shape[1] > 1:
        sns.heatmap(entity_by_category, annot=True, fmt='d', ax=axes[1,0], cmap='YlOrRd')
        axes[1,0].set_title('Entity Types by Category Heatmap')
    else:
        axes[1,0].text(0.5, 0.5, 'Insufficient data\nfor heatmap',
                      ha='center', va='center', transform=axes[1,0].transAxes)
        axes[1,0].set_title('Entity Types by Category')
    
    top_entities = entities_df['text'].value_counts().head(10)
    top_entities.plot(kind='barh', ax=axes[1,1])
    axes[1,1].set_title('Most Mentioned Entities')
    
    plt.tight_layout()
    plt.show()
    
    # Completed advanced entity analysis: co-occurrence, relationship-style pairs, time/date entities, and sentiment associations.
    print("\nADVANCED ENTITY ANALYSIS")
    print("=" * 50)
    
    pair_counts = Counter()
    for record in article_entities:
        selected_entities = sorted(set(
            entity['text'].strip()
            for entity in record['entities']
            if entity['label'] in ['PERSON', 'ORG', 'GPE', 'PRODUCT', 'EVENT'] and len(entity['text'].strip()) > 2
        ))[:12]
        pair_counts.update(combinations(selected_entities, 2))
    
    if pair_counts:
        cooccurrence_df = pd.DataFrame([
            {'entity_pair': f"{pair[0]} + {pair[1]}", 'co_mentions': count}
            for pair, count in pair_counts.most_common(15)
        ])
        print("\nTop entity co-occurrences:")
        print(cooccurrence_df)
        
        plt.figure(figsize=(10, 6))
        sns.barplot(data=cooccurrence_df, x='co_mentions', y='entity_pair')
        plt.title('Top Entity Co-occurrences')
        plt.xlabel('Articles mentioning both entities')
        plt.ylabel('Entity pair')
        plt.tight_layout()
        plt.show()
    else:
        cooccurrence_df = pd.DataFrame(columns=['entity_pair', 'co_mentions'])
        print("\nNo entity co-occurrences found in the selected entity types.")
    
    date_entities = entities_df[entities_df['label'] == 'DATE']['text'].value_counts().head(10)
    print("\nMost common DATE entities:")
    print(date_entities if not date_entities.empty else "No DATE entities found.")
    print("Note: the BBC dataset has no verified publication dates; the extra-credit section analyzes years mentioned in article text.")
    
    entity_sentiment = entities_df.merge(
        sentiment_df[['article_id', 'full_sentiment', 'full_label']],
        on='article_id',
        how='left'
    )
    entity_sentiment_summary = entity_sentiment.groupby(['text', 'label']).agg(
        mentions=('article_id', 'count'),
        avg_sentiment=('full_sentiment', 'mean')
    ).reset_index()
    entity_sentiment_summary = entity_sentiment_summary[entity_sentiment_summary['mentions'] >= 3]
    
    if not entity_sentiment_summary.empty:
        print("\nMost positive repeated entities:")
        print(entity_sentiment_summary.sort_values('avg_sentiment', ascending=False).head(10))
        print("\nMost negative repeated entities:")
        print(entity_sentiment_summary.sort_values('avg_sentiment', ascending=True).head(10))
    else:
        print("\nNot enough repeated entities for reliable entity-sentiment association.")
else:
    print("Skipping entity analysis due to insufficient data.")
    print("Tip: Try with a larger, more diverse dataset for better NER results.")


## Comprehensive Analysis and Insights

This section combines the project outputs into a single intelligence report: dataset quality, classification performance, sentiment trends, TF-IDF keywords, linguistic patterns, entity findings, and business recommendations.


In [ ]:
# Create comprehensive analysis dashboard
def create_comprehensive_analysis():
    """
    Generate a comprehensive NewsBot intelligence report.

    The report combines dataset quality, classifier performance, TF-IDF
    keywords, sentiment trends, entity findings, linguistic patterns, and
    business recommendations into one dictionary.
    """
    insights = {
        'dataset_overview': {},
        'classification_performance': {},
        'sentiment_insights': {},
        'entity_insights': {},
        'linguistic_patterns': {},
        'keyword_insights': {},
        'business_recommendations': []
    }
    
    insights['dataset_overview'] = {
        'total_articles': len(df),
        'categories': df['category'].unique().tolist(),
        'category_distribution': df['category'].value_counts().to_dict(),
        'avg_article_length': df['full_text'].str.len().mean(),
        'avg_words_per_article': df['full_text'].str.split().str.len().mean(),
        'missing_text_values': int(df['content'].isna().sum()),
        'missing_category_values': int(df['category'].isna().sum())
    }
    
    insights['classification_performance'] = {
        'best_model': best_model_name,
        'best_accuracy': results[best_model_name]['accuracy'],
        'best_cv_mean': results[best_model_name]['cv_mean'],
        'model_comparison': {name: results[name]['accuracy'] for name in results.keys()}
    }
    
    sentiment_by_cat = sentiment_df.groupby('category')['full_sentiment'].mean().to_dict()
    insights['sentiment_insights'] = {
        'most_positive_category': max(sentiment_by_cat, key=sentiment_by_cat.get),
        'most_negative_category': min(sentiment_by_cat, key=sentiment_by_cat.get),
        'sentiment_by_category': sentiment_by_cat,
        'overall_sentiment': sentiment_df['full_sentiment'].mean()
    }
    
    if not entities_df.empty:
        entity_by_cat = entities_df.groupby('category').size().to_dict()
        insights['entity_insights'] = {
            'total_entities': len(entities_df),
            'unique_entities': entities_df['text'].nunique(),
            'entity_types': entities_df['label'].unique().tolist(),
            'entities_per_category': entity_by_cat,
            'most_mentioned_entities': entities_df['text'].value_counts().head(5).to_dict()
        }
    
    if 'category_terms' in globals():
        insights['keyword_insights'] = {
            category: terms.head(5).index.tolist()
            for category, terms in category_terms.items()
        }
    
    if 'pos_interpretation' in globals():
        insights['linguistic_patterns']['pos_summary'] = pos_interpretation.round(4).to_dict()
    if 'syntax_summary' in globals():
        insights['linguistic_patterns']['syntax_summary'] = syntax_summary.round(2).to_dict()
    
    recommendations = []
    
    if insights['classification_performance']['best_accuracy'] > 0.8:
        recommendations.append("High classification accuracy supports automated content routing.")
    else:
        recommendations.append("Classifier is functional, but more tuning/data could improve automated routing reliability.")
    
    pos_cat = insights['sentiment_insights']['most_positive_category']
    neg_cat = insights['sentiment_insights']['most_negative_category']
    recommendations.append(f"{pos_cat} articles have the most positive average tone and may support uplifting content recommendations.")
    recommendations.append(f"{neg_cat} articles have the most negative average tone and may need closer editorial monitoring.")
    
    if insights.get('entity_insights'):
        recommendations.append("Entity extraction enables media monitoring, search, and relationship analysis by people, places, and organizations.")
    if insights.get('keyword_insights'):
        recommendations.append("TF-IDF keywords can be used to explain why the classifier associates articles with specific categories.")
    
    insights['business_recommendations'] = recommendations
    return insights

print("Generating comprehensive analysis...")
analysis_results = create_comprehensive_analysis()

print("Analysis complete!")
print("\n" + "=" * 60)
print("NEWSBOT INTELLIGENCE SYSTEM - COMPREHENSIVE REPORT")
print("=" * 60)

print("\nDATASET OVERVIEW:")
overview = analysis_results['dataset_overview']
print(f"  Total Articles: {overview['total_articles']}")
print(f"  Categories: {', '.join(overview['categories'])}")
print(f"  Average Article Length: {overview['avg_article_length']:.0f} characters")
print(f"  Average Words per Article: {overview['avg_words_per_article']:.0f} words")
print(f"  Missing Text Values: {overview['missing_text_values']}")
print(f"  Missing Category Values: {overview['missing_category_values']}")

print("\nCLASSIFICATION PERFORMANCE:")
perf = analysis_results['classification_performance']
print(f"  Best Model: {perf['best_model']}")
print(f"  Best Accuracy: {perf['best_accuracy']:.4f}")
print(f"  Best CV Mean: {perf['best_cv_mean']:.4f}")

print("\nSENTIMENT INSIGHTS:")
sent = analysis_results['sentiment_insights']
print(f"  Most Positive Category: {sent['most_positive_category']}")
print(f"  Most Negative Category: {sent['most_negative_category']}")
print(f"  Overall Sentiment: {sent['overall_sentiment']:.4f}")

if analysis_results.get('entity_insights'):
    print("\nENTITY INSIGHTS:")
    ent = analysis_results['entity_insights']
    print(f"  Total Entities: {ent['total_entities']}")
    print(f"  Unique Entities: {ent['unique_entities']}")
    print(f"  Entity Types: {', '.join(ent['entity_types'])}")

if analysis_results.get('keyword_insights'):
    print("\nKEYWORD INSIGHTS:")
    for category, terms in analysis_results['keyword_insights'].items():
        print(f"  {category}: {', '.join(terms)}")

print("\nBUSINESS RECOMMENDATIONS:")
for i, rec in enumerate(analysis_results['business_recommendations'], 1):
    print(f"  {i}. {rec}")


## Final System Integration

This section wraps the preprocessing, classification, entity extraction, sentiment analysis, and insight generation into one reusable NewsBot class. The final tests show how the bot handles new articles entered by a user.


In [ ]:
class NewsBotIntelligenceSystem:
    """
    Complete NewsBot Intelligence System
    
    💡 TIP: This class should encapsulate:
    - All preprocessing functions
    - Trained classification model
    - Entity extraction pipeline
    - Sentiment analysis
    - Insight generation
    """
    
    def __init__(self, classifier, vectorizer, sentiment_analyzer):
        """Store the trained model, TF-IDF vectorizer, sentiment analyzer, and spaCy pipeline."""
        self.classifier = classifier
        self.vectorizer = vectorizer
        self.sentiment_analyzer = sentiment_analyzer
        self.nlp = nlp  # spaCy model
    
    def preprocess_article(self, title, content):
        """Preprocess a new article"""
        full_text = f"{title} {content}"
        processed_text = preprocess_text(full_text)
        return full_text, processed_text
    
    def classify_article(self, title, full_text, processed_text):
        """Classify with the same TF-IDF, sentiment, and length features used for training."""
        tfidf_features = self.vectorizer.transform([processed_text])
        scores = analyze_sentiment(full_text)
        compound_scaled = (scores["compound"] + 1) / 2
        sentiment_features = np.array([[compound_scaled, scores["pos"], scores["neu"], scores["neg"]]])
        length_features = np.array([[len(full_text), len(full_text.split()), len(title)]], dtype=float)
        length_features = length_features / length_feature_max
        extra_features = csr_matrix(np.hstack([sentiment_features, length_features]))
        combined = hstack([tfidf_features, extra_features]).tocsr()
        
        prediction = self.classifier.predict(combined)[0]
        
        if hasattr(self.classifier, "predict_proba"):
            probabilities = self.classifier.predict_proba(combined)[0]
        else:
            # LinearSVC does not provide probabilities, so convert decision scores into
            # normalized confidence-like values for the dashboard display.
            scores_array = self.classifier.decision_function(combined)
            scores_array = np.ravel(scores_array)
            exp_scores = np.exp(scores_array - np.max(scores_array))
            probabilities = exp_scores / exp_scores.sum()
        
        return prediction, dict(zip(self.classifier.classes_, probabilities))

    def extract_entities(self, text):
        """Extract named entities"""
        return extract_entities(text)
    
    def analyze_sentiment(self, text):
        """Analyze sentiment"""
        return analyze_sentiment(text)
    
    def process_article(self, title, content):
        """
        Complete article processing pipeline
        
        💡 TIP: This should return a comprehensive analysis including:
        - Predicted category with confidence
        - Extracted entities
        - Sentiment analysis
        - Key insights and recommendations
        """
        # Run the completed NewsBot pipeline.
        
        # Step 1: Preprocess
        full_text, processed_text = self.preprocess_article(title, content)
        
        # Step 2: Classify
        category, category_probs = self.classify_article(title, full_text, processed_text)
        
        # Step 3: Extract entities
        entities = self.extract_entities(full_text)
        
        # Step 4: Analyze sentiment
        sentiment = self.analyze_sentiment(full_text)
        
        # Step 5: Generate insights
        insights = self.generate_insights(category, entities, sentiment, category_probs)
        
        return {
            'title': title,
            'content': content[:200] + '...' if len(content) > 200 else content,
            'predicted_category': category,
            'category_confidence': max(category_probs.values()),
            'category_probabilities': category_probs,
            'entities': entities,
            'sentiment': sentiment,
            'insights': insights
        }
    
    def generate_insights(self, category, entities, sentiment, category_probs):
        """Generate actionable insights"""
        insights = []
        
        # Classification insights
        confidence = max(category_probs.values())
        if confidence > 0.8:
            insights.append(f"✅ High confidence {category} classification ({confidence:.2%})")
        else:
            insights.append(f"⚠️ Uncertain classification - consider manual review")
        
        # Sentiment insights
        if sentiment['compound'] > 0.1:
            insights.append(f"😊 Positive sentiment detected ({sentiment['compound']:.3f})")
        elif sentiment['compound'] < -0.1:
            insights.append(f"😞 Negative sentiment detected ({sentiment['compound']:.3f})")
        else:
            insights.append(f"😐 Neutral sentiment ({sentiment['compound']:.3f})")
        
        # Entity insights
        if entities:
            entity_types = set([e['label'] for e in entities])
            insights.append(f"🔍 Found {len(entities)} entities of {len(entity_types)} types")
            
            # Highlight important entities
            important_entities = [e for e in entities if e['label'] in ['PERSON', 'ORG', 'GPE']]
            if important_entities:
                key_entities = [e['text'] for e in important_entities[:3]]
                insights.append(f"🎯 Key entities: {', '.join(key_entities)}")
        else:
            insights.append("ℹ️ No named entities detected")
        
        return insights

# Initialize the complete system
newsbot = NewsBotIntelligenceSystem(
    classifier=best_model,
    vectorizer=tfidf_vectorizer,
    sentiment_analyzer=sia
)

print("🤖 NewsBot Intelligence System initialized!")
print("✅ Ready to process new articles")


In [ ]:
# Test the complete system with new articles
print("TESTING NEWSBOT INTELLIGENCE SYSTEM")
print("=" * 60)

test_articles = [
    {
        'title': 'Microsoft Acquires AI Startup for $2 Billion',
        'content': "Microsoft Corporation announced today the acquisition of an artificial intelligence startup for $2 billion. CEO Satya Nadella said the deal will strengthen Microsoft's position in the AI market and enhance cloud computing services."
    },
    {
        'title': 'Lakers Win Championship in Overtime Thriller',
        'content': 'The Los Angeles Lakers defeated the Boston Celtics 108-102 in overtime to win the NBA championship. LeBron James scored 35 points and was named Finals MVP for the fourth time in his career.'
    },
    {
        'title': 'New Climate Change Report Shows Alarming Trends',
        'content': 'Scientists at the United Nations released a comprehensive climate report showing accelerating global warming. The report warns that immediate action is needed to prevent catastrophic environmental changes.'
    }
]

test_results = []

for i, article in enumerate(test_articles, 1):
    print(f"\nTEST ARTICLE {i}")
    print("-" * 40)
    
    result = newsbot.process_article(article['title'], article['content'])
    test_results.append({
        'title': article['title'],
        'predicted_category': result['predicted_category'],
        'confidence': result['category_confidence'],
        'sentiment': result['sentiment']['sentiment_label'],
        'entity_count': len(result['entities'])
    })
    
    print(f"Title: {result['title']}")
    print(f"Content: {result['content']}")
    print(f"\nPredicted Category: {result['predicted_category']} ({result['category_confidence']:.2%} confidence)")
    
    print("\nCategory Probabilities:")
    for cat, prob in sorted(result['category_probabilities'].items(), key=lambda x: x[1], reverse=True):
        print(f"  {cat}: {prob:.3f}")
    
    print(f"\nSentiment: {result['sentiment']['sentiment_label']} (score: {result['sentiment']['compound']:.3f})")
    
    if result['entities']:
        print(f"\nEntities Found ({len(result['entities'])}):")
        for entity in result['entities'][:5]:
            print(f"  {entity['text']} ({entity['label']}) - {entity['description']}")
    else:
        print("\nNo entities detected")
    
    print("\nInsights:")
    for insight in result['insights']:
        print(f"  {insight}")

print("\n" + "=" * 60)
print("NewsBot Intelligence System testing complete!")
print("System successfully processed all test articles")

print("\nSYSTEM TEST REFLECTION")
print("=" * 60)
test_results_df = pd.DataFrame(test_results)
print(test_results_df)
print("\nStrengths: the bot combines category prediction, sentiment, entity extraction, and explanation-style insights in one workflow.")
print("Limitations: the classifier can only predict the BBC training categories, so topics outside those labels are mapped to the closest available category.")
print("Improvement path: add newer articles, more categories such as health, and evaluate more custom NER examples.")


## Extra Credit Features

This organized section targets the available extra-credit opportunities:

- Interactive Dashboard (5 pts): a Gradio interface where a user can paste an article and receive category, sentiment, entities, and insights.
- Advanced Analysis (5 pts): temporal-reference trend analysis using years mentioned inside articles, because this BBC training file does not include verified publication dates.
- Custom Models (5 pts): domain-specific NER through a news entity ruler plus a small custom NER training demonstration.
- Research Extension (10 pts): confidence-aware NewsBot triage with quantitative evaluation.

### Extra Credit 1: Custom news-domain NER

The first extra-credit feature extends standard spaCy NER with news-specific labels such as TECH_COMPANY, SPORTS_LEAGUE, POLITICAL_ORG, and NEWS_SOURCE. The following cells also include a small custom NER training demonstration from hand-labeled examples.


In [ ]:
# Extra Credit: domain-specific NER with a transparent entity ruler.
if "news_domain_ruler" in nlp.pipe_names:
    nlp.remove_pipe("news_domain_ruler")

ruler = nlp.add_pipe("entity_ruler", name="news_domain_ruler", after="ner", config={"overwrite_ents": True})
ruler.add_patterns([
    {"label": "TECH_COMPANY", "pattern": "OpenAI"},
    {"label": "TECH_COMPANY", "pattern": "Microsoft"},
    {"label": "TECH_COMPANY", "pattern": "Google"},
    {"label": "TECH_COMPANY", "pattern": "Apple"},
    {"label": "NEWS_SOURCE", "pattern": "BBC"},
    {"label": "NEWS_SOURCE", "pattern": "Reuters"},
    {"label": "SPORTS_LEAGUE", "pattern": "NBA"},
    {"label": "SPORTS_LEAGUE", "pattern": "NFL"},
    {"label": "SPORTS_LEAGUE", "pattern": "FIFA"},
    {"label": "POLITICAL_ORG", "pattern": "White House"},
    {"label": "POLITICAL_ORG", "pattern": "United Nations"},
    {"label": "POLITICAL_ORG", "pattern": "Congress"}
])

custom_rule_demo = nlp("Reuters reported that OpenAI, Microsoft, the NBA, and the United Nations were mentioned in separate news stories.")
print("News-domain entity rules loaded.")
print("Rule-based custom NER demo:", [(ent.text, ent.label_) for ent in custom_rule_demo.ents])


In [ ]:
# Extra Credit: small custom NER training demonstration.
# This is intentionally small so it runs inside free Colab. It demonstrates the workflow
# for domain-specific NER training; a production model would need many more examples.
from spacy.training import Example
import random

train_data = [
    ("OpenAI released a research model.", {"entities": [(0, 6, "TECH_COMPANY")]}),
    ("Microsoft announced cloud investments.", {"entities": [(0, 9, "TECH_COMPANY")]}),
    ("Google launched a search feature.", {"entities": [(0, 6, "TECH_COMPANY")]}),
    ("Apple expanded its services business.", {"entities": [(0, 5, "TECH_COMPANY")]}),
    ("The NBA confirmed the playoff schedule.", {"entities": [(4, 7, "SPORTS_LEAGUE")]}),
    ("The NFL announced a broadcast deal.", {"entities": [(4, 7, "SPORTS_LEAGUE")]}),
    ("The White House released a statement.", {"entities": [(4, 15, "POLITICAL_ORG")]}),
    ("The United Nations held climate talks.", {"entities": [(4, 18, "POLITICAL_ORG")]}),
]

custom_ner_nlp = spacy.blank("en")
ner = custom_ner_nlp.add_pipe("ner")
for _, annotations in train_data:
    for _, _, label in annotations["entities"]:
        ner.add_label(label)

examples = [Example.from_dict(custom_ner_nlp.make_doc(text), ann) for text, ann in train_data]
optimizer = custom_ner_nlp.initialize(lambda: examples)

for _ in range(25):
    random.shuffle(examples)
    custom_ner_nlp.update(examples, sgd=optimizer, drop=0.2)

demo_texts = [
    "OpenAI and the NBA announced separate events.",
    "Microsoft and the United Nations appeared in the report.",
    "The NFL and Google signed media partnerships."
]

for demo_text in demo_texts:
    demo_doc = custom_ner_nlp(demo_text)
    print(demo_text, "->", [(entity.text, entity.label_) for entity in demo_doc.ents])


### Extra Credit 2: Advanced temporal-reference analysis

The BBC training file does not include verified publication dates. Instead of inventing dates, this analysis counts explicit four-digit years mentioned inside articles by category. This measures temporal references in coverage, not publication volume.


In [ ]:
# Extra Credit: temporal-reference trends from years mentioned in article text.
temporal_rows = []
for _, article in df.iterrows():
    for year in re.findall(r"\b(?:19|20)\d{2}\b", article["content"]):
        temporal_rows.append({"category": article["category"], "year_mentioned": int(year)})

temporal_df = pd.DataFrame(temporal_rows)

if temporal_df.empty:
    print("No four-digit years were found.")
else:
    temporal_counts = temporal_df.groupby(["year_mentioned", "category"]).size().reset_index(name="mentions")
    temporal_pivot = temporal_counts.pivot_table(
        index='year_mentioned', columns='category', values='mentions', fill_value=0
    )
    
    print("Temporal reference summary:")
    print(temporal_pivot.tail(15))
    
    print("\nMost frequently mentioned years:")
    print(temporal_df['year_mentioned'].value_counts().head(10))
    
    plt.figure(figsize=(11, 5))
    sns.lineplot(data=temporal_counts, x="year_mentioned", y="mentions", hue="category", marker="o")
    plt.title("Temporal References Mentioned in Articles by Category")
    plt.xlabel("Year mentioned inside article text")
    plt.ylabel("Mentions")
    plt.tight_layout()
    plt.show()


### Extra Credit 3: Interactive Gradio NewsBot dashboard

Run this after the final NewsBot test. The dashboard lets a user paste a headline and article text, then returns the predicted category, sentiment, named entities, and NewsBot insights.


In [ ]:
!pip -q install gradio
import gradio as gr

def dashboard_analyze(title, content):
    """
    Analyze user-provided article text for the Gradio dashboard.

    Returns category probabilities, sentiment, extracted entities, and
    generated NewsBot insights in dashboard-friendly formats.
    """
    try:
        result = newsbot.process_article(title, content)
        entities = "\n".join(f"- {x['text']} — {x['label']}" for x in result["entities"][:12]) or "No entities found."
        insights = "\n".join(f"- {x}" for x in result["insights"])
        sentiment = f"**{result['sentiment']['sentiment_label'].title()}** (VADER: {result['sentiment']['compound']:.2f})"
        return result["category_probabilities"], sentiment, entities, insights
    except Exception as error:
        return {}, f"Input issue: {error}", "", ""

with gr.Blocks(theme=gr.themes.Soft(), title="NewsBot") as demo:
    gr.Markdown("# NewsBot Intelligence System")
    gr.Markdown("Paste a news article to classify its category, sentiment, entities, and main insights.")
    title_input = gr.Textbox(label="Headline, optional")
    content_input = gr.Textbox(label="Article text", lines=12)
    button = gr.Button("Analyze article", variant="primary")
    category_output = gr.Label(label="Predicted category", num_top_classes=5)
    sentiment_output = gr.Markdown(label="Sentiment")
    entity_output = gr.Markdown(label="Entities")
    insight_output = gr.Markdown(label="Insights")
    button.click(dashboard_analyze, [title_input, content_input],
                 [category_output, sentiment_output, entity_output, insight_output])

demo.launch(share=False)


### Extra Credit 4: Research Extension — Confidence-Aware NewsBot Triage

Research question: Can NewsBot become more reliable by automatically classifying high-confidence articles while sending low-confidence articles to manual review?

This extension uses selective classification, also called confidence-aware triage. Instead of forcing every article into a category, the system evaluates different confidence thresholds and reports the tradeoff between automation coverage and accuracy.

Evaluation metrics:

- Auto coverage: percentage of test articles the system would classify automatically.
- Manual review rate: percentage of test articles flagged for review.
- Auto accuracy: accuracy only on articles above the confidence threshold.
- Effective accuracy with review: estimated system accuracy if reviewed articles are corrected by a human.


In [ ]:
# Research Extension: confidence-aware NewsBot triage with evaluation.
def get_prediction_confidence(model, X):
    """
    Return model predictions and confidence scores.

    Models with predict_proba use probability scores. LinearSVC does not
    provide probabilities, so decision scores are converted into softmax-style
    confidence values for evaluation.
    """
    predictions = model.predict(X)
    
    if hasattr(model, "predict_proba"):
        probabilities = model.predict_proba(X)
    else:
        decision_scores = model.decision_function(X)
        
        if decision_scores.ndim == 1:
            decision_scores = np.column_stack([-decision_scores, decision_scores])
        
        exp_scores = np.exp(decision_scores - np.max(decision_scores, axis=1, keepdims=True))
        probabilities = exp_scores / exp_scores.sum(axis=1, keepdims=True)
    
    confidence_scores = probabilities.max(axis=1)
    return predictions, confidence_scores


print("RESEARCH EXTENSION: CONFIDENCE-AWARE NEWSBOT TRIAGE")
print("=" * 70)
print("Research question:")
print("Can NewsBot improve reliability by auto-classifying confident articles and flagging uncertain articles for review?")

# Get predictions and confidence scores from the best model.
research_predictions, confidence_scores = get_prediction_confidence(best_model, X_test)
baseline_accuracy = accuracy_score(y_test, research_predictions)

print(f"\nBaseline test accuracy without triage: {baseline_accuracy:.4f}")

# Test several thresholds to evaluate the automation-versus-accuracy tradeoff.
thresholds = [0.00, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]
triage_rows = []

for threshold in thresholds:
    auto_mask = confidence_scores >= threshold
    auto_count = int(auto_mask.sum())
    total_count = len(confidence_scores)
    
    coverage = auto_count / total_count
    review_rate = 1 - coverage
    
    if auto_count > 0:
        auto_accuracy = accuracy_score(y_test[auto_mask], research_predictions[auto_mask])
        correct_auto = int((research_predictions[auto_mask] == y_test[auto_mask]).sum())
    else:
        auto_accuracy = np.nan
        correct_auto = 0
    
    # Assumption: manually reviewed articles would be corrected by a human.
    effective_accuracy_with_review = (correct_auto + (total_count - auto_count)) / total_count
    
    triage_rows.append({
        "confidence_threshold": threshold,
        "auto_classified_articles": auto_count,
        "auto_coverage": coverage,
        "manual_review_rate": review_rate,
        "auto_accuracy": auto_accuracy,
        "effective_accuracy_with_review": effective_accuracy_with_review
    })

triage_df = pd.DataFrame(triage_rows)

print("\nTriage evaluation results:")
display(triage_df.round(4))

# Pick a practical threshold: prefer at least 50% automation, then highest auto accuracy.
practical_options = triage_df[triage_df["auto_coverage"] >= 0.50].copy()
if not practical_options.empty:
    selected_threshold = practical_options.sort_values(
        by=["auto_accuracy", "auto_coverage"],
        ascending=False
    ).iloc[0]
else:
    selected_threshold = triage_df.sort_values(by="auto_accuracy", ascending=False).iloc[0]

print("\nSelected practical threshold:")
print(f"  Confidence threshold: {selected_threshold['confidence_threshold']:.2f}")
print(f"  Auto coverage: {selected_threshold['auto_coverage']:.1%}")
print(f"  Manual review rate: {selected_threshold['manual_review_rate']:.1%}")
print(f"  Auto accuracy: {selected_threshold['auto_accuracy']:.1%}")

# Visualize how accuracy and coverage change as the threshold increases.
plt.figure(figsize=(10, 5))
plt.plot(triage_df["confidence_threshold"], triage_df["auto_coverage"], marker="o", label="Auto coverage")
plt.plot(triage_df["confidence_threshold"], triage_df["auto_accuracy"], marker="o", label="Auto accuracy")
plt.plot(triage_df["confidence_threshold"], triage_df["manual_review_rate"], marker="o", label="Manual review rate")
plt.title("Research Extension: Confidence Threshold Tradeoff")
plt.xlabel("Confidence threshold")
plt.ylabel("Score")
plt.ylim(0, 1.05)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Show the lowest-confidence cases, which are the best candidates for manual review.
uncertain_cases = pd.DataFrame({
    "true_category": y_test,
    "predicted_category": research_predictions,
    "confidence": confidence_scores
}).sort_values("confidence").head(10)

print("\nLowest-confidence test cases:")
display(uncertain_cases.round(4))

print("\nResearch conclusion:")
print("Confidence-aware triage gives NewsBot a safer workflow: high-confidence articles can be routed automatically, while uncertain cases can be reviewed manually. This adds an evaluated human-in-the-loop approach instead of relying only on raw classifier accuracy.")


## 📝 Project Summary and Next Steps

### 🎯 What You've Accomplished

Congratulations! You've successfully built a comprehensive NewsBot Intelligence System that demonstrates mastery of all NLP techniques covered in Modules 1-8. Let's review what you've achieved:

### ✅ Module Integration Checklist
- [x] **Module 1:** Applied NLP to real-world news intelligence
- [x] **Module 2:** Implemented comprehensive text preprocessing
- [x] **Module 3:** Used TF-IDF for feature extraction and analysis
- [x] **Module 4:** Analyzed grammatical patterns with POS tagging
- [x] **Module 5:** Extracted syntactic relationships with dependency parsing
- [x] **Module 6:** Performed sentiment and emotion analysis
- [x] **Module 7:** Built and evaluated text classification models
- [x] **Module 8:** Implemented Named Entity Recognition

### 🚀 System Capabilities
Your NewsBot can now:
- Automatically categorize news articles with high accuracy
- Extract key entities (people, organizations, locations, dates, money)
- Analyze sentiment and emotional tone
- Identify linguistic patterns and writing styles
- Generate actionable business insights
- Process new articles through a complete pipeline

### 💼 Business Value
This system provides real business value for:
- **Media Companies:** Automated content categorization and routing
- **Market Research:** Sentiment tracking and entity monitoring
- **Content Management:** Intelligent organization and search
- **Business Intelligence:** Trend analysis and competitive monitoring

---

## 📋 Final Deliverables Checklist

Before submitting your project, ensure you have:

### 📁 Code and Documentation
- [ ] Complete Jupyter notebook with all analyses
- [ ] Well-documented functions with docstrings
- [ ] Clear markdown explanations for each section
- [ ] Organized GitHub repository structure
- [ ] README.md with project overview and setup instructions

### 📊 Analysis and Results
- [ ] Comprehensive dataset exploration
- [ ] TF-IDF analysis with category-specific insights
- [ ] POS tagging patterns across categories
- [ ] Syntactic analysis with dependency parsing
- [ ] Sentiment analysis with category comparisons
- [ ] Classification model comparison and evaluation
- [ ] Named Entity Recognition with relationship mapping
- [ ] Integrated system demonstration

### 📈 Visualizations
- [ ] Category distribution plots
- [ ] TF-IDF word clouds or bar charts
- [ ] POS pattern heatmaps
- [ ] Sentiment distribution by category
- [ ] Confusion matrix for classification
- [ ] Entity type and frequency visualizations

### 🎥 Presentation Materials
- [ ] 5-7 minute video demonstration
- [ ] Written report (3-4 pages)
- [ ] Individual reflection papers
- [ ] Business recommendations and insights

---

## 🔮 Future Enhancements

Consider these improvements for your portfolio or future projects:

### 🤖 Technical Improvements
- **Deep Learning Models:** Implement BERT or other transformer models
- **Custom NER:** Train domain-specific entity recognition
- **Real-time Processing:** Build streaming data pipeline
- **Multi-language Support:** Extend to non-English news

### 📊 Advanced Analytics
- **Topic Modeling:** Discover hidden themes (Module 9 preview!)
- **Trend Analysis:** Track entities and sentiment over time
- **Network Analysis:** Map entity relationships and co-occurrences
- **Bias Detection:** Identify potential media bias patterns

### 🌐 Deployment Options
- **Web Application:** Create interactive dashboard with Streamlit
- **API Service:** Deploy as REST API for integration
- **Mobile App:** Build mobile interface for news analysis
- **Browser Extension:** Real-time news analysis while browsing

---

## 🎓 Reflection Questions

For your individual reflection paper, consider these questions:

1. **Technical Mastery:** Which NLP techniques did you find most challenging? Most useful?
2. **Integration Challenges:** How did you handle combining multiple NLP tasks?
3. **Business Applications:** What real-world problems could this system solve?
4. **Ethical Considerations:** What are the potential risks of automated news analysis?
5. **Future Learning:** What NLP topics are you most excited to explore next?
6. **Team Collaboration:** How did you divide work and ensure quality?
7. **Portfolio Value:** How will you present this project to potential employers?

---

## 🏆 Congratulations!

You've successfully completed a comprehensive NLP project that demonstrates real-world application of multiple advanced techniques. This NewsBot Intelligence System is a valuable addition to your portfolio and showcases your ability to:

- **Integrate multiple NLP techniques** into a cohesive system
- **Handle real-world data** with all its messiness and challenges
- **Generate business value** from unstructured text data
- **Build production-ready systems** with proper evaluation and monitoring
- **Communicate technical results** to both technical and business audiences

**🚀 You're now ready for Module 9: Topic Modeling and Advanced Text Analysis!**

---

*Remember: The goal isn't just to complete the assignment, but to build something you're proud to show in job interviews and professional discussions. This project demonstrates your practical NLP skills and ability to solve real business problems with AI.*